# 🫀 Cardiac Ultrasound Segmentation — Inference Only

This notebook runs the **trained U-Net segmentation model** on new ultrasound images.

**You do NOT need the training dataset (database_nifti/) to use this notebook.**

### What you need:
- `segmentation_model.pt` — trained model weights (get from teammate)
- Your ultrasound images (`.nii.gz`, `.png`, `.jpg`, or `.dcm`)

### What this does:
1. Loads the trained segmentation model
2. Segments cardiac structures (LV cavity, myocardium, left atrium)
3. Calculates cardiac metrics (areas, volumes, EF, VTI, cardiac output)
4. Visualizes results with color overlays

In [6]:
# Cell [1] — Create a local virtual environment and install packages
import subprocess, sys, os

venv_path = os.path.expanduser("~/cardiac_venv")

# Create venv (only needs to run once)
if not os.path.exists(venv_path):
    subprocess.run([sys.executable, "-m", "venv", venv_path], check=True)
    print(f"✅ Created venv at {venv_path}")

# Install packages into the venv
pip_path = os.path.join(venv_path, "bin", "pip")
subprocess.run([pip_path, "install", "torch", "torchvision", "numpy", "nibabel", "matplotlib", "Pillow"], check=True)

# Add venv packages to this notebook's path
import site
site_packages = os.path.join(venv_path, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
site.addsitedir(site_packages)

print(f"\n✅ All packages installed and available!")
print(f"   Location: {site_packages}")

✅ Created venv at /home/users/leah26/cardiac_venv
  Using cached nibabel-5.3.3-py3-none-any.whl.metadata (9.1 kB)
  Using cached filelock-3.24.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.m


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /home/users/leah26/cardiac_venv/bin/python3.13 -m pip install --upgrade pip



✅ All packages installed and available!
   Location: /home/users/leah26/cardiac_venv/lib/python3.13/site-packages


In [7]:
# Cell [2] — Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision.transforms import functional as F
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


In [ ]:
# Cell [3] — Configuration

# Path to the trained model weights file CHANGE!!!!
MODEL_PATH = "checkpoints/segmentation_model.pt"

# Folder containing your ultrasound images to analyze CHANGE!!!!
IMAGE_DIR = "test_images/"

# Model input size (must match training)
IMG_SIZE = (256, 256)

# Segmentation label map
LABEL_MAP = {
    0: 'Background',
    1: 'LV Cavity',
    2: 'Myocardium',
    3: 'Left Atrium'
}

print("✅ Config loaded")
print(f"   Model path:  {MODEL_PATH}")
print(f"   Image dir:   {IMAGE_DIR}")
print(f"   Input size:  {IMG_SIZE}")

In [ ]:
# Cell [4] — U-Net Model Architecture
# (Must match exactly what was used during training)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4):
        super(UNet, self).__init__()
        
        # Encoder
        self.enc1 = self._conv_block(in_channels, 64)
        self.enc2 = self._conv_block(64, 128)
        self.enc3 = self._conv_block(128, 256)
        self.enc4 = self._conv_block(256, 512)
        
        # Bottleneck
        self.bottleneck = self._conv_block(512, 1024)
        
        # Decoder
        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = self._conv_block(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self._conv_block(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self._conv_block(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self._conv_block(128, 64)
        
        self.out = nn.Conv2d(64, out_channels, 1)
        self.pool = nn.MaxPool2d(2)
        
    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e4))
        
        # Decoder with skip connections
        d4 = torch.cat([self.upconv4(b), e4], dim=1)
        d4 = self.dec4(d4)
        d3 = torch.cat([self.upconv3(d4), e3], dim=1)
        d3 = self.dec3(d3)
        d2 = torch.cat([self.upconv2(d3), e2], dim=1)
        d2 = self.dec2(d2)
        d1 = torch.cat([self.upconv1(d2), e1], dim=1)
        d1 = self.dec1(d1)
        
        return self.out(d1)

print(f"✅ UNet architecture defined ({sum(p.numel() for p in UNet().parameters()):,} parameters)")

In [ ]:
# Cell [5] — Load Trained Model

def load_model(path=MODEL_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"❌ Model not found at: {path}\n"
            f"   Get 'segmentation_model.pt' from your teammate and update MODEL_PATH in Cell [3]"
        )
    
    model = UNet().to(DEVICE)
    model.load_state_dict(
        torch.load(path, map_location=DEVICE, weights_only=True)
    )
    model.eval()
    print(f"✅ Model loaded from: {path}")
    return model

model = load_model()

In [ ]:
# Cell [6] — Image Loading Helpers
# Supports multiple formats your ultrasound hardware might output

def load_ultrasound_image(path):
    """
    Load an ultrasound image from various formats.
    Returns: (image_array, pixel_spacing)
        image_array: 2D numpy array (H, W)
        pixel_spacing: (dx, dy) in mm, or (1.0, 1.0) if unknown
    """
    path = str(path)
    ext = path.lower()
    
    if ext.endswith(('.nii', '.nii.gz')):
        # NIfTI format (CAMUS dataset format)
        import nibabel as nib
        nii = nib.load(path)
        img = nii.get_fdata().astype(np.float32)
        # Handle 3D volumes — take middle slice if needed
        if img.ndim == 3:
            img = img[:, :, img.shape[2] // 2]
        spacing = nii.header.get_zooms()[:2]
        return img, spacing
    
    elif ext.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        # Standard image formats
        from PIL import Image
        img = np.array(Image.open(path).convert('L')).astype(np.float32)
        return img, (1.0, 1.0)  # Unknown pixel spacing
    
    elif ext.endswith('.dcm'):
        # DICOM format
        try:
            import pydicom
            ds = pydicom.dcmread(path)
            img = ds.pixel_array.astype(np.float32)
            spacing = getattr(ds, 'PixelSpacing', [1.0, 1.0])
            return img, (float(spacing[0]), float(spacing[1]))
        except ImportError:
            raise ImportError("Install pydicom: pip install pydicom")
    
    elif ext.endswith('.npy'):
        # NumPy array
        img = np.load(path).astype(np.float32)
        return img, (1.0, 1.0)
    
    else:
        raise ValueError(f"Unsupported format: {path}")

print("✅ Image loader ready")
print("   Supported formats: .nii.gz, .nii, .png, .jpg, .dcm, .npy")

In [ ]:
# Cell [7] — Segmentation Inference

def predict_segmentation(model, image_array):
    """
    Run segmentation on a single 2D ultrasound image.
    
    Args:
        model: loaded UNet model
        image_array: 2D numpy array (H, W)
    
    Returns:
        pred_mask: 2D numpy array (H, W) with labels 0-3
        confidence: 2D numpy array (H, W) with prediction confidence
    """
    original_shape = image_array.shape[:2]
    
    # Normalize to [0, 1]
    img = image_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    
    # Convert to tensor: (1, 1, H, W)
    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    img_tensor = img_tensor.to(DEVICE)
    
    # Run inference
    with torch.no_grad():
        output = model(img_tensor)                     # (1, 4, 256, 256)
        probs = torch.softmax(output, dim=1)           # probabilities
        confidence, pred = torch.max(probs, dim=1)     # (1, 256, 256)
    
    # Resize back to original size
    pred = pred.unsqueeze(1).float()
    pred = F.resize(pred, original_shape, interpolation=F.InterpolationMode.NEAREST)
    
    confidence = confidence.unsqueeze(1).float()
    confidence = F.resize(confidence, original_shape, interpolation=F.InterpolationMode.BILINEAR)
    
    return (
        pred.squeeze().cpu().numpy().astype(np.int64),
        confidence.squeeze().cpu().numpy()
    )

print("✅ Segmentation function ready")

In [ ]:
# Cell [8] — Cardiac Metrics Calculation

def calculate_areas(pred_mask, pixel_spacing_mm=(1.0, 1.0)):
    """Calculate areas of each segmented structure in mm²."""
    dx, dy = pixel_spacing_mm
    pixel_area = dx * dy
    
    return {
        'lv_area_mm2': float((pred_mask == 1).sum() * pixel_area),
        'myocardium_area_mm2': float((pred_mask == 2).sum() * pixel_area),
        'la_area_mm2': float((pred_mask == 3).sum() * pixel_area),
    }


def calculate_lv_length_mm(pred_mask, pixel_spacing_mm=(1.0, 1.0)):
    """Estimate LV long-axis length from segmentation mask."""
    dx, dy = pixel_spacing_mm
    ys, xs = np.where(pred_mask == 1)
    
    if len(xs) == 0:
        return None
    
    length_pixels = np.sqrt((xs.max() - xs.min())**2 + (ys.max() - ys.min())**2)
    return float(length_pixels * np.mean([dx, dy]))


def calculate_volume_biplane(area_2ch_mm2, area_4ch_mm2, length_mm):
    """
    Biplane area-length method (modified Simpson's):
    V = (8 / 3π) × (A_2CH × A_4CH) / L
    Returns volume in mL.
    """
    if None in [area_2ch_mm2, area_4ch_mm2, length_mm] or length_mm == 0:
        return None
    
    volume_mm3 = (8 / (3 * np.pi)) * (area_2ch_mm2 * area_4ch_mm2) / length_mm
    return volume_mm3 / 1000  # mm³ → mL


def calculate_ef(edv_ml, esv_ml):
    """Ejection Fraction (%)."""
    if edv_ml is None or esv_ml is None or edv_ml == 0:
        return None
    return ((edv_ml - esv_ml) / edv_ml) * 100


def calculate_cardiac_output(sv_ml, heart_rate_bpm):
    """Cardiac Output in L/min."""
    return (sv_ml * heart_rate_bpm) / 1000


def calculate_vti(sv_ml, lvot_diameter_cm=2.0):
    """
    Velocity Time Integral (cm).
    VTI = SV / LVOT_area
    """
    lvot_area = np.pi * (lvot_diameter_cm / 2)**2
    return sv_ml / lvot_area  # mL = cm³, so result is in cm


print("✅ Cardiac metrics functions ready")

In [ ]:
# Cell [9] — Visualization

def visualize_result(image, pred_mask, confidence, title=""):
    """Display original image, segmentation, and overlay."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # Original
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Segmentation mask
    colors = plt.cm.get_cmap('tab10', 4)
    axes[1].imshow(pred_mask, cmap='tab10', vmin=0, vmax=3)
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    
    # Overlay
    axes[2].imshow(image, cmap='gray')
    for label, color, name in [(1, 'Reds', 'LV'), (2, 'Blues', 'Myo'), (3, 'Greens', 'LA')]:
        mask_overlay = np.ma.masked_where(pred_mask != label, pred_mask)
        axes[2].imshow(mask_overlay, cmap=color, alpha=0.5, vmin=0, vmax=3)
    axes[2].set_title('Overlay (R=LV, B=Myo, G=LA)')
    axes[2].axis('off')
    
    # Confidence map
    im = axes[3].imshow(confidence, cmap='RdYlGn', vmin=0.5, vmax=1.0)
    axes[3].set_title(f'Confidence (avg: {confidence.mean():.2f})')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046)
    
    if title:
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()

print("✅ Visualization functions ready")

In [ ]:
# Cell [10] — Run on a Single Image

# UPDATE THIS PATH to your ultrasound image


test_image_path = "test_images/your_ultrasound.nii.gz"  # <-- CHANGE THIS

if os.path.exists(test_image_path):
    # Load image
    image, spacing = load_ultrasound_image(test_image_path)
    print(f"📷 Loaded: {test_image_path}")
    print(f"   Shape: {image.shape}, Pixel spacing: {spacing} mm")
    
    # Segment
    pred_mask, confidence = predict_segmentation(model, image)
    
    # Calculate areas
    areas = calculate_areas(pred_mask, spacing)
    lv_length = calculate_lv_length_mm(pred_mask, spacing)
    
    print(f"\n📊 Segmentation Results:")
    print(f"   LV Cavity Area:    {areas['lv_area_mm2']:.1f} mm²")
    print(f"   Myocardium Area:   {areas['myocardium_area_mm2']:.1f} mm²")
    print(f"   Left Atrium Area:  {areas['la_area_mm2']:.1f} mm²")
    if lv_length:
        print(f"   LV Length:         {lv_length:.1f} mm")
    print(f"   Avg Confidence:    {confidence.mean():.2f}")
    
    # Visualize
    visualize_result(image, pred_mask, confidence, title=Path(test_image_path).name)
else:
    print(f"⚠️  File not found: {test_image_path}")
    print(f"   Update 'test_image_path' above to point to your ultrasound image")

In [ ]:
# Cell [11] — Full Pipeline: Process a Set of 15 Probe Images

# This processes all images from your hardware's 15 poses


def process_all_images(image_dir, model):
    """
    Process all ultrasound images in a directory.
    Returns a list of results for each image.
    """
    supported_ext = ('.nii.gz', '.nii', '.png', '.jpg', '.jpeg', '.dcm', '.npy')
    
    # Find all image files
    image_files = []
    for f in sorted(os.listdir(image_dir)):
        if any(f.lower().endswith(ext) for ext in supported_ext):
            image_files.append(os.path.join(image_dir, f))
    
    if not image_files:
        print(f"❌ No supported images found in {image_dir}")
        return []
    
    print(f"📂 Found {len(image_files)} images in {image_dir}\n")
    
    results = []
    for i, img_path in enumerate(image_files, 1):
        print(f"Processing [{i}/{len(image_files)}]: {Path(img_path).name}")
        
        try:
            image, spacing = load_ultrasound_image(img_path)
            pred_mask, confidence = predict_segmentation(model, image)
            areas = calculate_areas(pred_mask, spacing)
            lv_length = calculate_lv_length_mm(pred_mask, spacing)
            
            results.append({
                'file': Path(img_path).name,
                'image': image,
                'pred_mask': pred_mask,
                'confidence': confidence,
                'spacing': spacing,
                'areas': areas,
                'lv_length_mm': lv_length,
                'avg_confidence': float(confidence.mean()),
                'has_lv': bool((pred_mask == 1).any()),
                'has_myocardium': bool((pred_mask == 2).any()),
                'has_la': bool((pred_mask == 3).any()),
            })
            print(f"   ✅ LV={areas['lv_area_mm2']:.0f}mm² | Confidence={confidence.mean():.2f}")
            
        except Exception as e:
            print(f"   ❌ Error: {e}")
            results.append({'file': Path(img_path).name, 'error': str(e)})
    
    return results

# Run on all images
if os.path.isdir(IMAGE_DIR):
    all_results = process_all_images(IMAGE_DIR, model)
else:
    print(f"⚠️  Directory not found: {IMAGE_DIR}")
    print(f"   Create the folder and add your ultrasound images, or update IMAGE_DIR in Cell [3]")
    all_results = []

In [ ]:
# Cell [12] — Calculate Volumes, EF, VTI, and Cardiac Output

# Requires 2CH and 4CH views from ED and ES phases
# Adjust the mapping below to match YOUR image filenames

def compute_full_cardiac_report(results, heart_rate_bpm=75, lvot_diameter_cm=2.0):
    """
    Given results from process_all_images(), compute full cardiac metrics.
    
    You need to identify which images correspond to:
      - 2CH ED (2-chamber, end-diastole)
      - 2CH ES (2-chamber, end-systole)
      - 4CH ED (4-chamber, end-diastole)
      - 4CH ES (4-chamber, end-systole)
    """
    valid = [r for r in results if 'error' not in r]
    
    if not valid:
        print("❌ No valid results to analyze")
        return None
    
    print("\n" + "=" * 60)
    print("🫀 CARDIAC REPORT")
    print("=" * 60)
    
    # Per-image summary
    print(f"\n📋 Individual Image Results ({len(valid)} images):")
    print(f"{'File':<35} {'LV mm²':>8} {'Myo mm²':>8} {'LA mm²':>8} {'Conf':>6}")
    print("-" * 70)
    for r in valid:
        print(f"{r['file']:<35} "
              f"{r['areas']['lv_area_mm2']:>8.1f} "
              f"{r['areas']['myocardium_area_mm2']:>8.1f} "
              f"{r['areas']['la_area_mm2']:>8.1f} "
              f"{r['avg_confidence']:>6.2f}")
    
    # Try to compute volumes if we can identify 2CH/4CH ED/ES 
    print("\n" + "-" * 60)
    print("💡 To compute volumes, EF, VTI, and cardiac output,")
    print("   identify your 2CH/4CH ED/ES images below:\n")
    
    # Example: manually specify which results correspond to which views
    # Uncomment and update these indices based on your images:
    #
    # idx_2ch_ed = 0   # index in 'valid' list for 2-chamber end-diastole
    # idx_2ch_es = 1   # index for 2-chamber end-systole
    # idx_4ch_ed = 2   # index for 4-chamber end-diastole  
    # idx_4ch_es = 3   # index for 4-chamber end-systole
    #
    # a2ch_ed = valid[idx_2ch_ed]['areas']['lv_area_mm2']
    # a4ch_ed = valid[idx_4ch_ed]['areas']['lv_area_mm2']
    # l_ed = valid[idx_4ch_ed]['lv_length_mm']
    # edv = calculate_volume_biplane(a2ch_ed, a4ch_ed, l_ed)
    #
    # a2ch_es = valid[idx_2ch_es]['areas']['lv_area_mm2']
    # a4ch_es = valid[idx_4ch_es]['areas']['lv_area_mm2']
    # l_es = valid[idx_4ch_es]['lv_length_mm']
    # esv = calculate_volume_biplane(a2ch_es, a4ch_es, l_es)
    #
    # if edv and esv:
    #     sv = edv - esv
    #     ef = calculate_ef(edv, esv)
    #     co = calculate_cardiac_output(sv, heart_rate_bpm)
    #     vti = calculate_vti(sv, lvot_diameter_cm)
    #
    #     print(f"  EDV:             {edv:.1f} mL")
    #     print(f"  ESV:             {esv:.1f} mL")
    #     print(f"  Stroke Volume:   {sv:.1f} mL")
    #     print(f"  Ejection Frac:   {ef:.1f} %")
    #     print(f"  Cardiac Output:  {co:.2f} L/min (at {heart_rate_bpm} bpm)")
    #     print(f"  VTI:             {vti:.1f} cm (LVOT Ø = {lvot_diameter_cm} cm)")
    
    print("\n   Uncomment the section above and set the correct indices!")
    
    return valid

if all_results:
    report = compute_full_cardiac_report(all_results, heart_rate_bpm=75, lvot_diameter_cm=2.0)

In [ ]:
# Cell [13] — Visualize All Results in a Grid

def visualize_all_results(results, cols=5):
    """Display all segmentation results in a grid."""
    valid = [r for r in results if 'error' not in r]
    
    if not valid:
        print("No valid results to display")
        return
    
    n = len(valid)
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1) if cols > 1 else np.array([[axes]])
    
    for i, r in enumerate(valid):
        row, col = i // cols, i % cols
        ax = axes[row, col]
        
        ax.imshow(r['image'], cmap='gray')
        for label, cmap_name in [(1, 'Reds'), (2, 'Blues'), (3, 'Greens')]:
            overlay = np.ma.masked_where(r['pred_mask'] != label, r['pred_mask'])
            ax.imshow(overlay, cmap=cmap_name, alpha=0.5, vmin=0, vmax=3)
        
        ax.set_title(f"{r['file']}\nConf: {r['avg_confidence']:.2f}", fontsize=9)
        ax.axis('off')
    
    # Hide empty subplots
    for i in range(n, rows * cols):
        row, col = i // cols, i % cols
        axes[row, col].axis('off')
    
    plt.suptitle("Segmentation Results — All Images", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

if all_results:
    visualize_all_results(all_results)